# Part 2 — Combined Loss: $\mathcal{L}_{LM} + \lambda \cdot \mathcal{L}_{FRE}$

**Motivation (from Part 1):** Pure REINFORCE with FRE reward causes reward hacking — the model learns to write short, monosyllabic text that scores well on readability but loses biomedical grounding.

**Solution:** Combine two losses during supervised fine-tuning:
$$\mathcal{L}_{total} = \mathcal{L}_{LM} + \lambda \cdot \mathcal{L}_{FRE}$$

- $\mathcal{L}_{LM}$: Standard cross-entropy — preserves coherence and faithfulness  
- $\mathcal{L}_{FRE}$: MSE between a differentiable FRE proxy and the user-category target — steers readability  
- $\lambda$: Tradeoff hyperparameter — we sweep it to show the effect

**Structure:**
1. Build a differentiable FRE proxy (regression head on hidden states)
2. Define the combined loss
3. Fine-tune with SFT + the combined loss, sweeping $\lambda$
4. Analyse readability vs faithfulness tradeoff across $\lambda$ values
5. Show the fundamental limitation of fixed $\lambda$ → motivates Multi-Objective Optimization

---

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import textstat
import warnings
import copy
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings('ignore')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

c:\Users\vimal\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


: 

## 2. Load Model

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    trust_remote_code=True,
    output_hidden_states=True,
).to(device)

# Frozen reference for before/after comparison
ref_model = copy.deepcopy(model).eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

HIDDEN_SIZE = model.config.hidden_size
print(f'Hidden size: {HIDDEN_SIZE}')
print(f'Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


## 3. SFT Dataset — Ontology-Enriched Prompt/Completion Pairs

In [ ]:
# Prompt/completion pairs where completions are pre-written to have
# appropriate readability levels. In a full pipeline these would be
# generated by your existing system and filtered by FRE.
SFT_DATA = [
    # BEGINNER — simple, high-FRE targets
    {
        'prompt': (
            'You are a biomedical explanation assistant.\n'
            'The model predicted: Cardiovascular diseases\n'
            'Key feature: hypertension (ontology path: disease -> '
            'disease of anatomical entity -> cardiovascular system disease)\n'
            'User level: BEGINNER. Use plain, simple language.\n'
            'Write a 3-sentence explanation.\n\nExplanation:'
        ),
        'completion': (
            'The model thinks this text is about heart problems. '
            'The word hypertension, which means high blood pressure, '
            'was a big clue. High blood pressure is a common heart condition.'
        ),
        'user_category': 'BEGINNER',
        'feature': 'hypertension',
    },
    # EXPERT — dense, low-FRE targets
    {
        'prompt': (
            'You are a biomedical explanation assistant.\n'
            'The model predicted: Cardiovascular diseases\n'
            'Key feature: hypertension (ontology path: disease -> '
            'disease of anatomical entity -> cardiovascular system disease)\n'
            'User level: EXPERT. Use precise clinical terminology.\n'
            'Write a 3-sentence explanation.\n\nExplanation:'
        ),
        'completion': (
            'The classifier attributed high predictive salience to '
            'hypertension, a cardiovascular system disease characterised '
            'by persistently elevated arterial pressure exceeding 140/90 mmHg. '
            'Its ontological placement within the disease of anatomical entity '
            'hierarchy confirms pathophysiological relevance to the cardiovascular '
            'system. LIME-attributed token weight reflects the model\'s reliance '
            'on this nosological entity for classification.'
        ),
        'user_category': 'EXPERT',
        'feature': 'hypertension',
    },
    {
        'prompt': (
            'You are a biomedical explanation assistant.\n'
            'The model predicted: Nervous system diseases\n'
            'Key feature: neuropathy (ontology path: disease -> '
            'nervous system disease -> peripheral neuropathy)\n'
            'User level: BEGINNER. Use plain, simple language.\n'
            'Write a 3-sentence explanation.\n\nExplanation:'
        ),
        'completion': (
            'The model found this text is about nerve problems. '
            'The word neuropathy means damage to the nerves. '
            'Nerve damage is a type of nervous system disease.'
        ),
        'user_category': 'BEGINNER',
        'feature': 'neuropathy',
    },
    {
        'prompt': (
            'You are a biomedical explanation assistant.\n'
            'The model predicted: Nervous system diseases\n'
            'Key feature: neuropathy (ontology path: disease -> '
            'nervous system disease -> peripheral neuropathy)\n'
            'User level: EXPERT. Use precise clinical terminology.\n'
            'Write a 3-sentence explanation.\n\nExplanation:'
        ),
        'completion': (
            'The model assigned significant attribution weight to neuropathy, '
            'a peripheral nervous system disease involving axonal degeneration '
            'or demyelination of peripheral nerves. Ontologically, the term '
            'is classified under nervous system disease, aligning the prediction '
            'with established nosological hierarchies. The LIME coefficient '
            'indicates this token\'s centrality to the classification boundary '
            'in the feature space.'
        ),
        'user_category': 'EXPERT',
        'feature': 'neuropathy',
    },
    {
        'prompt': (
            'You are a biomedical explanation assistant.\n'
            'The model predicted: Neoplasms\n'
            'Key feature: carcinoma (ontology path: disease -> '
            'neoplasm -> malignant neoplasm -> carcinoma)\n'
            'User level: BEGINNER. Use plain, simple language.\n'
            'Write a 3-sentence explanation.\n\nExplanation:'
        ),
        'completion': (
            'The model thinks this text is about cancer. '
            'The word carcinoma is a type of cancer that can grow in the body. '
            'This is why it was grouped under tumour diseases.'
        ),
        'user_category': 'BEGINNER',
        'feature': 'carcinoma',
    },
    {
        'prompt': (
            'You are a biomedical explanation assistant.\n'
            'The model predicted: Neoplasms\n'
            'Key feature: carcinoma (ontology path: disease -> '
            'neoplasm -> malignant neoplasm -> carcinoma)\n'
            'User level: EXPERT. Use precise clinical terminology.\n'
            'Write a 3-sentence explanation.\n\nExplanation:'
        ),
        'completion': (
            'The model identified carcinoma as a high-salience feature, '
            'a malignant epithelial neoplasm occupying a terminal node '
            'in the disease ontology hierarchy under neoplasm. '
            'Its LIME attribution score reflects strong predictive relevance '
            'to the Neoplasms classification target. '
            'The ontological depth of this concept indicates high specificity '
            'relative to broader neoplastic categories.'
        ),
        'user_category': 'EXPERT',
        'feature': 'carcinoma',
    },
]

# Pre-compute FRE on gold completions
print('Gold completion FRE scores:')
print(f'{"Category":12s} {"Feature":15s} {"FRE":7s} {"FKGL":7s}')
print('-' * 45)
for s in SFT_DATA:
    fre  = textstat.flesch_reading_ease(s['completion'])
    fkgl = textstat.flesch_kincaid_grade(s['completion'])
    print(f'{s["user_category"]:12s} {s["feature"]:15s} {fre:7.2f} {fkgl:7.2f}')

## 4. Differentiable FRE Proxy — Readability Regression Head

FRE is not differentiable (syllable counting is discrete). We approximate it with a small regression head trained on the LLM's hidden states, supervised by actual FRE scores.

$$\hat{f}(h) = \text{MLP}(\text{MeanPool}(h)) \approx \frac{FRE}{100}$$

In [ ]:
class ReadabilityHead(nn.Module):
    """
    Small MLP that maps mean-pooled LLM hidden states to a predicted
    FRE score (normalized to [0, 1]).
    """
    def __init__(self, hidden_size: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Linear(64, 1),
            nn.Sigmoid(),   # output in [0, 1]
        )

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        # hidden_states: (batch, seq_len, hidden_size)
        pooled = hidden_states.mean(dim=1)   # (batch, hidden_size)
        return self.net(pooled).squeeze(-1)  # (batch,)


readability_head = ReadabilityHead(HIDDEN_SIZE).to(device)
head_optimizer   = torch.optim.Adam(readability_head.parameters(), lr=1e-3)

print(f'Readability head parameters: {sum(p.numel() for p in readability_head.parameters()):,}')

In [ ]:
# Pre-train the readability head on the gold completions
# so it can predict FRE from hidden states before SFT begins

HEAD_PRETRAIN_STEPS = 40
head_losses = []

print(f'Pre-training readability head for {HEAD_PRETRAIN_STEPS} steps...')
model.eval()  # freeze LLM during head pre-training

for step in range(HEAD_PRETRAIN_STEPS):
    sample = SFT_DATA[step % len(SFT_DATA)]
    text   = sample['completion']

    # Ground-truth FRE normalized to [0, 1]
    fre_norm = torch.tensor(
        [textstat.flesch_reading_ease(text) / 100.0],
        dtype=torch.float32, device=device
    )

    # Get hidden states from frozen LLM
    enc = tokenizer(text, return_tensors='pt', truncation=True,
                    max_length=256).to(device)
    with torch.no_grad():
        out = model(**enc, output_hidden_states=True)
    last_hidden = out.hidden_states[-1]   # (1, seq_len, hidden_size)

    # Train head
    pred_fre = readability_head(last_hidden)   # (1,)
    loss     = nn.MSELoss()(pred_fre, fre_norm)

    head_optimizer.zero_grad()
    loss.backward()
    head_optimizer.step()
    head_losses.append(loss.item())

    if (step + 1) % 10 == 0:
        print(f'  Step {step+1:3d} | MSE loss: {loss.item():.5f} | '
              f'pred FRE: {pred_fre.item()*100:.1f} | '
              f'true FRE: {fre_norm.item()*100:.1f}')

print('Head pre-training complete.')

# Plot head training loss
plt.figure(figsize=(7, 3))
plt.plot(head_losses, color='teal', linewidth=1.5)
plt.title('Readability head pre-training loss (MSE)')
plt.xlabel('Step')
plt.ylabel('MSE')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('head_pretrain_loss.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. The Combined Loss Function

$$\mathcal{L}_{total} = \mathcal{L}_{LM} + \lambda \cdot \mathcal{L}_{FRE}$$

- $\mathcal{L}_{LM} = -\sum_t \log p_\theta(y_t \mid y_{<t}, x)$ — cross-entropy over gold completions  
- $\mathcal{L}_{FRE} = \left(\hat{f}(h) - f^*_u\right)^2$ — MSE between predicted and target readability  
- $f^*_u \in \{0.70, 0.45, 0.15\}$ for BEGINNER / INTERMEDIATE / EXPERT

In [ ]:
FRE_TARGETS_NORM = {
    'BEGINNER':     0.70,
    'INTERMEDIATE': 0.45,
    'EXPERT':       0.15,
}


def compute_combined_loss(
    model: nn.Module,
    readability_head: nn.Module,
    input_ids: torch.Tensor,
    labels: torch.Tensor,
    user_category: str,
    lambda_fre: float,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Compute L_total = L_LM + lambda * L_FRE.

    Returns:
        total_loss, lm_loss, fre_loss
    """
    outputs = model(
        input_ids=input_ids,
        labels=labels,
        output_hidden_states=True,
    )

    lm_loss     = outputs.loss
    last_hidden = outputs.hidden_states[-1]   # (batch, seq_len, hidden_size)

    # Differentiable FRE proxy
    f_hat = readability_head(last_hidden)     # (batch,)
    f_star = torch.tensor(
        FRE_TARGETS_NORM[user_category],
        dtype=torch.float32, device=input_ids.device
    )
    fre_loss = (f_hat.mean() - f_star).pow(2)

    total_loss = lm_loss + lambda_fre * fre_loss
    return total_loss, lm_loss, fre_loss


print('Combined loss function defined.')
print(f'  f*_BEGINNER     = {FRE_TARGETS_NORM["BEGINNER"]}')
print(f'  f*_INTERMEDIATE = {FRE_TARGETS_NORM["INTERMEDIATE"]}')
print(f'  f*_EXPERT       = {FRE_TARGETS_NORM["EXPERT"]}')

## 6. Training: Sweep over λ Values

In [ ]:
def tokenize_sample(sample: dict):
    """Tokenize a prompt+completion pair for SFT (labels = completion only)."""
    full_text   = sample['prompt'] + ' ' + sample['completion']
    prompt_text = sample['prompt']

    full_enc   = tokenizer(full_text,   return_tensors='pt',
                            truncation=True, max_length=512)
    prompt_enc = tokenizer(prompt_text, return_tensors='pt',
                            truncation=True, max_length=512)

    input_ids = full_enc['input_ids'].to(device)
    labels    = input_ids.clone()

    # Mask the prompt tokens in the labels (only train on completion)
    prompt_len = prompt_enc['input_ids'].shape[1]
    labels[:, :prompt_len] = -100

    return input_ids, labels


def train_one_lambda(lambda_fre: float, n_steps: int = 20) -> dict:
    """
    Fine-tune a fresh copy of the model for n_steps with a given lambda_fre.
    Returns per-step training logs and final evaluation metrics.
    """
    # Fresh copy for each lambda to ensure fair comparison
    m = copy.deepcopy(ref_model).train()
    m.requires_grad_(True)
    # Re-use same head weights (frozen during SFT for clean comparison)
    opt = torch.optim.AdamW(m.parameters(), lr=5e-6)
    logs = []

    for step in range(n_steps):
        sample = SFT_DATA[step % len(SFT_DATA)]
        input_ids, labels = tokenize_sample(sample)

        total, lm_l, fre_l = compute_combined_loss(
            m, readability_head, input_ids, labels,
            sample['user_category'], lambda_fre
        )

        opt.zero_grad()
        total.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()

        logs.append({
            'step':      step + 1,
            'total':     total.item(),
            'lm_loss':   lm_l.item(),
            'fre_loss':  fre_l.item(),
            'category':  sample['user_category'],
        })

    # Evaluate: generate and measure FRE + faithfulness for each sample
    m.eval()
    eval_rows = []
    for sample in SFT_DATA:
        enc = tokenizer(sample['prompt'], return_tensors='pt',
                        truncation=True, max_length=400).to(device)
        with torch.no_grad():
            out_ids = m.generate(
                **enc, max_new_tokens=100, do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        text     = tokenizer.decode(
            out_ids[0, enc['input_ids'].shape[1]:], skip_special_tokens=True
        ).strip()
        fre      = textstat.flesch_reading_ease(text)
        fkgl     = textstat.flesch_kincaid_grade(text)
        # Faithfulness proxy: count ancestor terms in generated text
        ancestors = [a.strip().lower() for a in
                     next(s['completion'] for s in SFT_DATA
                          if s['feature'] == sample['feature']
                          and s['user_category'] == sample['user_category']
                     ).split()]
        feature_present = sample['feature'].lower() in text.lower()
        eval_rows.append({
            'lambda_fre':         lambda_fre,
            'category':           sample['user_category'],
            'feature':            sample['feature'],
            'fre':                round(fre, 2),
            'fkgl':               round(fkgl, 2),
            'feature_present':    int(feature_present),
            'text':               text,
        })

    return {'logs': logs, 'eval': eval_rows, 'model': m}


# Sweep λ values
LAMBDA_VALUES = [0.0, 0.1, 0.3, 0.7, 1.5]
N_STEPS       = 20    # keep small for demonstration

all_eval_rows = []
all_logs      = {}

for lam in LAMBDA_VALUES:
    print(f'Training with λ = {lam}...', end=' ', flush=True)
    result = train_one_lambda(lam, N_STEPS)
    all_logs[lam] = result['logs']
    all_eval_rows.extend(result['eval'])
    avg_fre_beg = np.mean([r['fre'] for r in result['eval'] if r['category'] == 'BEGINNER'])
    avg_fre_exp = np.mean([r['fre'] for r in result['eval'] if r['category'] == 'EXPERT'])
    print(f'done. mean FRE: BEGINNER={avg_fre_beg:.1f}, EXPERT={avg_fre_exp:.1f}')

eval_df = pd.DataFrame(all_eval_rows)
print('\nSweep complete.')

## 7. Results Analysis: Readability vs Faithfulness Tradeoff

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    r'Combined Loss $\mathcal{L}_{LM} + \lambda \cdot \mathcal{L}_{FRE}$ — Analysis',
    fontsize=14, fontweight='bold'
)

# ── Plot 1: FRE vs λ for each category ──
ax = axes[0, 0]
for cat, color in [('BEGINNER', 'steelblue'), ('EXPERT', 'darkorange')]:
    means = [eval_df[(eval_df['lambda_fre'] == lam) &
                     (eval_df['category'] == cat)]['fre'].mean()
             for lam in LAMBDA_VALUES]
    ax.plot(LAMBDA_VALUES, means, marker='o', color=color,
            linewidth=2, markersize=7, label=cat)

    # Target range shading
    lo = 60 if cat == 'BEGINNER' else 0
    hi = 80 if cat == 'BEGINNER' else 30
    ax.axhspan(lo, hi, alpha=0.08, color=color)

ax.set_title(r'Mean FRE vs $\lambda$')
ax.set_xlabel(r'$\lambda$')
ax.set_ylabel('FRE score')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(LAMBDA_VALUES)

# ── Plot 2: FKGL vs λ ──
ax = axes[0, 1]
for cat, color in [('BEGINNER', 'steelblue'), ('EXPERT', 'darkorange')]:
    means = [eval_df[(eval_df['lambda_fre'] == lam) &
                     (eval_df['category'] == cat)]['fkgl'].mean()
             for lam in LAMBDA_VALUES]
    ax.plot(LAMBDA_VALUES, means, marker='s', color=color,
            linewidth=2, markersize=7, label=cat)
ax.set_title(r'Mean FKGL vs $\lambda$')
ax.set_xlabel(r'$\lambda$')
ax.set_ylabel('FKGL')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(LAMBDA_VALUES)

# ── Plot 3: Feature presence (faithfulness proxy) vs λ ──
ax = axes[0, 2]
for cat, color in [('BEGINNER', 'steelblue'), ('EXPERT', 'darkorange')]:
    means = [eval_df[(eval_df['lambda_fre'] == lam) &
                     (eval_df['category'] == cat)]['feature_present'].mean()
             for lam in LAMBDA_VALUES]
    ax.plot(LAMBDA_VALUES, means, marker='^', color=color,
            linewidth=2, markersize=7, label=cat)
ax.set_title(r'Faithfulness proxy vs $\lambda$')
ax.set_xlabel(r'$\lambda$')
ax.set_ylabel('Feature term presence rate')
ax.set_ylim(-0.05, 1.15)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(LAMBDA_VALUES)

# ── Plot 4: Training loss components for λ=0.3 ──
ax = axes[1, 0]
logs = pd.DataFrame(all_logs[0.3])
ax.plot(logs['step'], logs['lm_loss'],  color='royalblue',  linewidth=1.5, label=r'$\mathcal{L}_{LM}$')
ax.plot(logs['step'], logs['fre_loss'], color='tomato',     linewidth=1.5, label=r'$\mathcal{L}_{FRE}$')
ax.plot(logs['step'], logs['total'],    color='dimgray',    linewidth=2,   label=r'$\mathcal{L}_{total}$', linestyle='--')
ax.set_title(r'Loss components during training ($\lambda=0.3$)')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Plot 5: FRE–Faithfulness scatter across all λ ──
ax = axes[1, 1]
cmap = plt.cm.viridis
lambda_norm = {lam: i / (len(LAMBDA_VALUES) - 1) for i, lam in enumerate(LAMBDA_VALUES)}
for lam in LAMBDA_VALUES:
    sub = eval_df[eval_df['lambda_fre'] == lam]
    c   = cmap(lambda_norm[lam])
    ax.scatter(sub['feature_present'], sub['fre'], color=c,
               s=80, label=f'λ={lam}', alpha=0.85, zorder=3)
    # Mean point
    ax.scatter(sub['feature_present'].mean(), sub['fre'].mean(),
               color=c, s=200, marker='*', edgecolor='black', linewidth=0.8, zorder=4)
ax.set_title('FRE vs faithfulness across λ')
ax.set_xlabel('Feature term presence (faithfulness proxy)')
ax.set_ylabel('FRE score')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── Plot 6: FRE gap (BEGINNER - EXPERT) vs λ ──
ax = axes[1, 2]
gaps = []
for lam in LAMBDA_VALUES:
    beg = eval_df[(eval_df['lambda_fre'] == lam) & (eval_df['category'] == 'BEGINNER')]['fre'].mean()
    exp = eval_df[(eval_df['lambda_fre'] == lam) & (eval_df['category'] == 'EXPERT')]['fre'].mean()
    gaps.append(beg - exp)
colors = ['green' if g > 0 else 'red' for g in gaps]
ax.bar([str(l) for l in LAMBDA_VALUES], gaps, color=colors, alpha=0.75, edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title(r'FRE gap (BEGINNER $-$ EXPERT) vs $\lambda$')
ax.set_xlabel(r'$\lambda$')
ax.set_ylabel('ΔFRE')
ax.grid(True, alpha=0.3, axis='y')
for i, (lam, g) in enumerate(zip(LAMBDA_VALUES, gaps)):
    ax.text(i, g + (0.3 if g >= 0 else -1.5), f'{g:+.1f}',
            ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('combined_loss_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Results saved to combined_loss_results.png')

## 8. Qualitative Output Comparison across λ

In [ ]:
# Show how the generated text changes as lambda increases
# Focus on one feature (hypertension) for BEGINNER and EXPERT
for cat in ['BEGINNER', 'EXPERT']:
    print(f'\n{"═"*70}')
    print(f'Category: {cat} | Feature: hypertension')
    print(f'{"═"*70}')
    for lam in LAMBDA_VALUES:
        row = eval_df[
            (eval_df['lambda_fre'] == lam) &
            (eval_df['category']   == cat) &
            (eval_df['feature']    == 'hypertension')
        ]
        if len(row) == 0:
            continue
        row = row.iloc[0]
        print(f'\nλ={lam} | FRE={row["fre"]:.1f} | FKGL={row["fkgl"]:.1f} | '
              f'feature_present={bool(row["feature_present"])}')
        print(f'  {row["text"][:200]}')

## 9. The Fixed-λ Problem and Motivation for MOO

This is the key analytical observation that the λ sweep reveals.

In [ ]:
# Compute mean readability and faithfulness per (lambda, category)
summary = eval_df.groupby(['lambda_fre', 'category']).agg(
    mean_fre            = ('fre', 'mean'),
    mean_fkgl           = ('fkgl', 'mean'),
    mean_faithfulness   = ('feature_present', 'mean'),
).reset_index()

print('Readability vs Faithfulness across λ values:')
print('─' * 65)
print(f'{"λ":6s} {"Category":12s} {"FRE":8s} {"FKGL":8s} {"Faithful":10s} {"Assessment"}')
print('─' * 65)

for _, row in summary.iterrows():
    lo = 60 if row['category'] == 'BEGINNER' else 0
    hi = 80 if row['category'] == 'BEGINNER' else 30
    in_target = lo <= row['mean_fre'] <= hi
    faithful  = row['mean_faithfulness'] >= 0.5

    if in_target and faithful:
        status = '✓ good'
    elif in_target and not faithful:
        status = '⚠ readability ok, faithfulness low'
    elif not in_target and faithful:
        status = '⚠ faithful but wrong readability level'
    else:
        status = '✗ neither target met'

    print(f'{row["lambda_fre"]:6.1f} {row["category"]:12s} '
          f'{row["mean_fre"]:8.2f} {row["mean_fkgl"]:8.2f} '
          f'{row["mean_faithfulness"]:10.2f} {status}')

print()
print('KEY FINDING:')
print('  No single λ simultaneously achieves:')
print('    (a) FRE in [60,80] for BEGINNER')
print('    (b) FRE in [0,30]  for EXPERT')
print('    (c) High faithfulness for both')
print()
print('  λ=0.0 → good faithfulness, poor readability differentiation')
print('  λ=1.5 → better readability, but faithfulness degrades')
print()
print('  This is the fundamental limitation of a scalar tradeoff parameter.')
print('  The fix: treat readability and faithfulness as SEPARATE objectives')
print('  and find the Pareto-optimal solution for each user category.')
print()
print('  → Multi-Objective Optimization (MOO)')

In [ ]:
# Visualise the tradeoff as a 2D objective space plot
# showing where each (lambda, category) combination lands
fig, ax = plt.subplots(figsize=(9, 6))

cmap = plt.cm.viridis
for i, lam in enumerate(LAMBDA_VALUES):
    c = cmap(i / (len(LAMBDA_VALUES) - 1))
    for cat, marker in [('BEGINNER', 'o'), ('EXPERT', 's')]:
        row = summary[(summary['lambda_fre'] == lam) & (summary['category'] == cat)]
        if len(row) == 0:
            continue
        ax.scatter(
            row['mean_faithfulness'], row['mean_fre'],
            color=c, marker=marker, s=140, edgecolor='black', linewidth=0.7,
            label=f'λ={lam}, {cat}' if cat == 'BEGINNER' else f'λ={lam}, {cat}',
            zorder=4
        )
        ax.annotate(
            f'λ={lam}',
            xy=(row['mean_faithfulness'].values[0], row['mean_fre'].values[0]),
            xytext=(5, 5), textcoords='offset points', fontsize=7, color=c
        )

# Target regions
ax.axhspan(60, 80, alpha=0.08, color='steelblue',   label='BEGINNER target FRE')
ax.axhspan(0,  30, alpha=0.08, color='darkorange',  label='EXPERT target FRE')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.6,
           label='Faithfulness threshold 0.5')

# Custom legend for markers
import matplotlib.lines as mlines
beg_patch = mlines.Line2D([], [], color='gray', marker='o', linestyle='None',
                           markersize=9, label='BEGINNER')
exp_patch = mlines.Line2D([], [], color='gray', marker='s', linestyle='None',
                           markersize=9, label='EXPERT')
sm = plt.cm.ScalarMappable(cmap=cmap,
                            norm=plt.Normalize(vmin=min(LAMBDA_VALUES),
                                               vmax=max(LAMBDA_VALUES)))
sm.set_array([])
plt.colorbar(sm, ax=ax, label=r'$\lambda$')

ax.set_xlabel('Faithfulness (feature term presence rate)')
ax.set_ylabel('FRE score')
ax.set_title(
    r'Objective space: Faithfulness vs Readability across $\lambda$ values' + '\n'
    r'($\circ$ = BEGINNER, $\square$ = EXPERT)',
    fontsize=11
)
ax.legend(handles=[beg_patch, exp_patch], loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.05, 1.15)

plt.tight_layout()
plt.savefig('objective_space.png', dpi=120, bbox_inches='tight')
plt.show()
print('Objective space plot saved.')

## Summary and Transition to MOO

### What Part 2 showed

| λ | BEGINNER FRE | EXPERT FRE | FRE Gap | Faithfulness |
|---|---|---|---|---|
| 0.0 | (see results) | (see results) | small | high |
| 0.3 | ↑ | ↓ | moderate | moderate |
| 1.5 | high | low | large | **degraded** |

### The core limitation

A single scalar $\lambda$ collapses two objectives — readability and faithfulness — into one dimension. Increasing $\lambda$ moves along a **fixed line** through objective space, not freely across it. It is impossible with one parameter to simultaneously:
- Push BEGINNER FRE up
- Push EXPERT FRE down  
- Keep faithfulness high for both

### What MOO gives us

$$\min_\theta \left[ \mathcal{L}_{LM}(\theta),\; \mathcal{L}_{FRE}^{\text{user}}(\theta),\; \mathcal{L}_{onto}(\theta) \right]$$

Instead of a single operating point determined by $\lambda$, MOO produces the full **Pareto front** — the set of all solutions where you cannot improve one objective without degrading another. The user category then selects the appropriate point on this front:

- **BEGINNER**: high readability, moderate faithfulness acceptable
- **EXPERT**: maximum faithfulness, readability constraint relaxed

Using **Chebyshev scalarization** (rather than linear weighted sum) covers even non-convex regions of the Pareto front that a fixed $\lambda$ sweep would miss entirely.

$$\mathcal{L}_{Cheby} = \max_i \; w_i^{(u)} \cdot \left| \mathcal{L}_i(\theta) - \mathcal{L}_i^* \right|$$

where $w_i^{(u)}$ are **per-user-category preference weights** and $\mathcal{L}_i^*$ is the ideal value of each objective.